# HDF5 to CSV Converter

Convert experiment HDF5 files to CSV for external analysis tools.

In [15]:
import os, glob, csv, datetime
import numpy as np
import h5py
from pathlib import Path

# Pin THIS checkout's results dir explicitly.
# (The old version imported analysis.data_loader, which on this machine resolved
#  to a DIFFERENT checkout and made RESULTS_DIR point at the wrong folder, so
#  every RID lookup failed with "No HDF5 for RID ...". No import needed here.)
RESULTS_DIR = Path(r"C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\results")


def load_h5(filepath):
    """Read an ARTIQ HDF5 result file into {data, metadata, archive}.

    locking=False so it works even while the experiment is still writing the
    file (handy for checking a live run mid-acquisition).
    """
    data, metadata, archive = {}, {}, {}
    with h5py.File(str(filepath), "r", locking=False) as f:
        if "datasets" in f:
            for k in f["datasets"]:
                data[k] = f["datasets"][k][()]
            for k in f["datasets"].attrs:
                data[k] = f["datasets"].attrs[k]
        if "metadata" in f:
            for k in f["metadata"].attrs:
                metadata[k] = f["metadata"].attrs[k]
        if "archive" in f:
            for k in f["archive"]:
                archive[k] = f["archive"][k][()]
            for k in f["archive"].attrs:
                archive[k] = f["archive"].attrs[k]
    return {"data": data, "metadata": metadata, "archive": archive}


print("Results directory:", RESULTS_DIR)
print("Exists:", RESULTS_DIR.exists())
print("Ready.")

Results directory: C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\results
Exists: True
Ready.


## 1. Select File

In [16]:
########################################
RID = 596   # Set RID, or None for latest
########################################

if RID is not None:
    pattern = "%09d-*.h5" % RID
    matches = glob.glob(str(RESULTS_DIR / "**" / pattern), recursive=True)
    if not matches:
        raise FileNotFoundError("No HDF5 for RID %d under %s" % (RID, RESULTS_DIR))
    filepath = max(matches, key=os.path.getmtime)
else:
    all_files = sorted(glob.glob(str(RESULTS_DIR / "**" / "*.h5"), recursive=True),
                       key=os.path.getmtime)
    if not all_files:
        raise FileNotFoundError("No HDF5 files under %s" % RESULTS_DIR)
    filepath = all_files[-1]

print("File:", os.path.basename(filepath))
print("Path:", filepath)

result = load_h5(filepath)
data = result["data"]
meta = result["metadata"]
archive = result["archive"]

print("\nDatasets (%d):" % len(data))
for k in sorted(data.keys()):
    v = data[k]
    if isinstance(v, np.ndarray):
        print("  %-35s  %s  shape=%s" % (k, v.dtype, v.shape))
    else:
        val = v.decode() if isinstance(v, bytes) else v
        print("  %-35s  = %s" % (k, val))

File: 000000596-ExperimentVariables.h5
Path: C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\results\2026-06-11\000000596-ExperimentVariables.h5

Datasets (131):
  HWP_angle_max                        = 360.0
  HWP_angle_min                        = 0.0
  HWP_angle_step                       = 30.0
  QWP_angle_max                        = 360.0
  QWP_angle_min                        = 0.0
  QWP_angle_step                       = 30.0
  SPCM1_counts                         = 0
  SPCM2_counts                         = 0
  amp_D1_onoff_L                       = 0.43
  amp_D1_pump                          = 0.1
  amp_FORT                             = 0.24
  amp_MOT_X                            = 0.12
  amp_MOT_Y                            = 0.1
  amp_MOT_Z                            = 0.08
  amp_RF                               = 0.1
  amp_RF_2ph                           = 0.6
  amp_cooling_DP_idle                  = 0.45
  amp_cooling_MOT                      = 0.3
  am

## 2. Choose Datasets to Export

Set which array datasets to include as CSV columns. Scalar values go into the header.

In [17]:
########################################
# Which array datasets to export as columns (set to None for all arrays)
COLUMNS = None
# Or pick specific ones:
# COLUMNS = ["mw_scan_x", "mw_retention", "mw_atom_loaded", "mw_atom_retained",
#            "mw_shot1_spcm1", "mw_shot1_spcm2", "mw_shot2_spcm1", "mw_shot2_spcm2",
#            "mw_bg_spcm1", "mw_bg_spcm2", "mw_load_attempts"]
########################################

# Auto-detect: export all arrays with the same length
if COLUMNS is None:
    # Find the most common array length (the per-shot data)
    lengths = {}
    for k, v in data.items():
        if isinstance(v, np.ndarray) and v.ndim == 1 and v.size > 1:
            lengths.setdefault(v.size, []).append(k)
    if lengths:
        main_len = max(lengths, key=lambda l: len(lengths[l]))
        COLUMNS = sorted(lengths[main_len])
        print("Auto-detected %d columns with %d rows:" % (len(COLUMNS), main_len))
    else:
        COLUMNS = []
        print("No array datasets found")
else:
    print("Using %d specified columns" % len(COLUMNS))

for c in COLUMNS:
    print("  %s" % c)

No array datasets found


## 3. Export to CSV

In [18]:
# Output path: same name as the H5 but .csv extension
csv_path = filepath.replace(".h5", ".csv")

# Header comments: run metadata + every scalar dataset (e.g. snapshot_*, exc_best_amp)
header_lines = ["# Source: %s" % os.path.basename(filepath)]
for k, v in sorted(meta.items()):
    val = v.decode() if isinstance(v, bytes) else v
    header_lines.append("# %s: %s" % (k, val))
for k in sorted(data.keys()):
    v = data[k]
    if not isinstance(v, np.ndarray) or v.ndim == 0:
        if isinstance(v, np.ndarray):
            val = v.item()
        else:
            val = v.decode() if isinstance(v, bytes) else v
        header_lines.append("# %s: %s" % (k, val))

# --- Main CSV: the per-measurement columns (COLUMNS, all same length) ---
arrays, n_rows = {}, None
for col in COLUMNS:
    if col in data and isinstance(data[col], np.ndarray) and data[col].ndim == 1:
        arr = data[col]
        if n_rows is None:
            n_rows = len(arr)
        if len(arr) == n_rows:
            arrays[col] = arr
        else:
            print("WARNING: %s has %d rows (expected %d), skipping" % (col, len(arr), n_rows))
    else:
        print("WARNING: %s not found or not a 1-D array, skipping" % col)

if not arrays:
    print("No per-measurement data to export!")
else:
    cols = sorted(arrays.keys())
    with open(csv_path, "w", newline="") as f:
        for line in header_lines:
            f.write(line + "\n")
        writer = csv.writer(f)
        writer.writerow(cols)
        for i in range(n_rows):
            writer.writerow([arrays[c][i] for c in cols])
    print("Main CSV : %d rows x %d cols -> %s" % (n_rows, len(cols), os.path.basename(csv_path)))
    print("Columns  : %s" % ", ".join(cols))

# --- Extra CSVs: any OTHER 1-D array length-groups (e.g. per-photon SPCM*_timestamps) ---
extra = {}
for k, v in data.items():
    if isinstance(v, np.ndarray) and v.ndim == 1 and v.size > 1 and k not in arrays:
        extra.setdefault(v.size, []).append(k)
for L, keys in sorted(extra.items()):
    keys = sorted(keys)
    extra_path = filepath.replace(".h5", "_len%d.csv" % L)
    with open(extra_path, "w", newline="") as f:
        f.write("# Source: %s  (arrays of length %d)\n" % (os.path.basename(filepath), L))
        writer = csv.writer(f)
        writer.writerow(keys)
        for i in range(L):
            writer.writerow([data[c][i] for c in keys])
    print("Extra CSV: %d rows x %d cols -> %s  (%s)" % (L, len(keys), os.path.basename(extra_path), ", ".join(keys)))

No per-measurement data to export!


## 4. Export Archive (calibration datasets) to separate CSV

In [19]:
########################################
EXPORT_ARCHIVE = True  # Set False to skip
########################################

if EXPORT_ARCHIVE and archive:
    archive_csv = filepath.replace(".h5", "_archive.csv")
    with open(archive_csv, 'w', newline='') as f:
        f.write("# Archive (persistent calibration datasets)\n")
        f.write("# Source: %s\n" % os.path.basename(filepath))
        writer = csv.writer(f)
        writer.writerow(["key", "value"])
        for k in sorted(archive.keys()):
            v = archive[k]
            if isinstance(v, bytes):
                v = v.decode()
            elif isinstance(v, np.ndarray):
                v = v.item() if v.ndim == 0 else v.tolist()
            writer.writerow([k, v])
    print("Archive exported (%d keys):" % len(archive))
    print("  %s" % archive_csv)
else:
    print("Archive export skipped.")

Archive export skipped.
